# Charset EDA: ASCII separator for HN titles

Hacker News titles are English → effectively ASCII (bytes `0x00`–`0x7F`). The
tokenizer uses a byte-level BPE pre-tokenizer (`train.py`, `ByteLevel`), which
operates over the full 256-byte space. If titles barely touch bytes
`0x80`–`0xFF`, most of that space is wasted allocation.

The current record separator is the multi-char string `<eos>`
(`train.py:238`), joined between titles. This notebook checks whether a
**single ASCII control byte** could replace it instead — specifically one of
the block `0x1C`–`0x1F`: FS/GS/RS/US, literally "File/Group/Record/Unit
Separator". These were designed for exactly this job, are dead in modern
text, are pure ASCII (backward compatible if the model ever falls back to
raw ASCII handling), and carry near-zero collision risk with real content.

This is analysis only — it does not modify `train.py`.

In [1]:
# Load titles the same way train.py does (train.py:82-86), so the charset
# scan reflects exactly what training sees.
from datasets import load_dataset

num_titles, seed, val_frac = 100_000, 1337, 0.10
ds = load_dataset("julien040/hacker-news-posts", split="train", cache_dir="../data").shuffle(seed=seed)
titles = [row["title"].strip() for row in ds.take(num_titles)]

print(f"loaded {len(titles)} titles")
print(titles[:5])

loaded 100000 titles
['WMS – Weather, Moon, & Solar / Weather Management System. TUI', "The web's broken deal with AI companies", 'Racket version 8.11 is now available', 'Omertà', 'Show HN: Interactive graphs in Rerun with a Rust port of D3-force']


In [2]:
# Enumerate every byte that actually appears in the titles once encoded as
# UTF-8 -- this is the same byte space the ByteLevel pre-tokenizer sees.
from collections import Counter

all_bytes = Counter()
for t in titles:
    all_bytes.update(t.encode("utf-8"))

total_bytes = sum(all_bytes.values())
printable = range(0x20, 0x7f)

print(f"{'byte':>6} {'hex':>6} {'count':>8}")
for b in sorted(all_bytes):
    label = chr(b) if b in printable else bytes([b])
    print(f"{label!r:>6} {hex(b):>6} {all_bytes[b]:>8}")

ascii_bytes = sum(c for b, c in all_bytes.items() if b < 0x80)
non_ascii_bytes = total_bytes - ascii_bytes
unused = 256 - len(all_bytes)

print()
print(f"distinct byte values used : {len(all_bytes)} / 256 ({len(all_bytes)/256:.1%})")
print(f"ASCII bytes (<0x80)        : {ascii_bytes} / {total_bytes} ({ascii_bytes/total_bytes:.4%})")
print(f"non-ASCII bytes (>=0x80)   : {non_ascii_bytes} / {total_bytes} ({non_ascii_bytes/total_bytes:.4%})")
print(f"unused byte values         : {unused} ({unused/256:.1%}) -- free budget for separators")

  byte    hex    count
   ' '   0x20   731647
   '!'   0x21       86
   '"'   0x22     3662
   '#'   0x23      208
   '$'   0x24     1736
   '%'   0x25      875
   '&'   0x26      268
   "'"   0x27    17251
   '('   0x28     7378
   ')'   0x29     7387
   '*'   0x2a       84
   '+'   0x2b      948
   ','   0x2c    14124
   '-'   0x2d    19427
   '.'   0x2e     9231
   '/'   0x2f     2279
   '0'   0x30    14613
   '1'   0x31     9434
   '2'   0x32    14807
   '3'   0x33     4839
   '4'   0x34     4226
   '5'   0x35     4555
   '6'   0x36     2836
   '7'   0x37     2122
   '8'   0x38     2379
   '9'   0x39     2954
   ':'   0x3a    25110
   ';'   0x3b      191
   '<'   0x3c       66
   '='   0x3d       84
   '>'   0x3e       90
   '?'   0x3f     7851
   '@'   0x40        9
   'A'   0x41    50962
   'B'   0x42    19598
   'C'   0x43    36948
   'D'   0x44    22023
   'E'   0x45    16757
   'F'   0x46    16347
   'G'   0x47    16703
   'H'   0x48    27251
   'I'   0x49    35914
   'J'   0x

## Rationale

The charset scan above confirms titles are almost entirely ASCII: the wide
byte-level vocab is mostly unused headroom. That headroom is exactly where a
dedicated separator byte should live, and ASCII control chars `0x1C`–`0x1F`
are the standard answer:

- **Purpose-built**: defined in ASCII specifically as record/field/group
  separators — not repurposed punctuation.
- **Obsolete in practice**: no modern text uses them, so collision risk with
  real title content is effectively zero.
- **ASCII backward compatible**: single byte, no multi-byte encoding
  ambiguity, resilient if anything downstream treats the stream as plain
  ASCII instead of the tokenizer's vocab.
- **Cheaper than `<eos>`**: one byte/token vs. five characters for the same
  boundary marker.

In [3]:
# Verify none of the four candidate separator bytes ever occur in a title.
candidates = {"FS": 0x1c, "GS": 0x1d, "RS": 0x1e, "US": 0x1f}

print(f"{'name':>4} {'hex':>6} {'present':>8} {'count':>8}")
for name, b in candidates.items():
    count = all_bytes.get(b, 0)
    print(f"{name:>4} {hex(b):>6} {str(count > 0):>8} {count:>8}")
    assert count == 0, f"{name} (0x{b:02x}) appears in titles -- not safe to use as separator"

print("\nPASS: all four control-char separators are absent from every title")

name    hex  present    count
  FS   0x1c    False        0
  GS   0x1d    False        0
  RS   0x1e    False        0
  US   0x1f    False        0

PASS: all four control-char separators are absent from every title


In [4]:
# Casing breakdown -- feeds the two-separator proposal below.
caps = sum(1 for t in titles if t.isupper())
title_case = sum(1 for t in titles if t.istitle())
other = len(titles) - caps - title_case
n = len(titles)

print(f"ALL_CAPS   : {caps:>7} ({caps/n:.2%})")
print(f"Title Case : {title_case:>7} ({title_case/n:.2%})")
print(f"other      : {other:>7} ({other/n:.2%})")

ALL_CAPS   :      69 (0.07%)
Title Case :    9740 (9.74%)
other      :   90191 (90.19%)


## Proposal

Verified above: FS (`0x1C`), GS (`0x1D`), RS (`0x1E`), US (`0x1F`) all occur
zero times in the title set. Any of them is safe to use as a record
separator; two of them can double as a casing signal at the same time as
marking the boundary:

- **First free separator (e.g. FS, `0x1C`)** before an **ALL_CAPS** title.
- **Second free separator (e.g. GS, `0x1D`)** before a **Title Case** title.
- Remaining titles keep a plain separator (e.g. RS, `0x1E`), or fold into
  one of the two above — to decide once wired in.

This packs the record boundary and a casing hint into a single byte/token,
instead of the current 5-char `<eos>` string.